# LH Nautical — Análise Exploratória de Dados

## Configuração do ambiente


In [2]:
from pathlib import Path

import duckdb
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

conn = duckdb.connect()

orders_path = RAW_DATA_DIR / "orders.csv"

conn.execute(
    f"""
    CREATE OR REPLACE VIEW orders AS
    SELECT *
    FROM read_csv_auto('{orders_path.as_posix()}');
    """
)

## Questão 1 — Análise exploratória da tabela `orders`

### Objetivo

Realizar uma análise exploratória inicial da tabela `orders`, avaliando seu
volume, período disponível, distribuição da variável `total` e possíveis
problemas de qualidade.

### Premissas

- Utilizar exclusivamente a tabela `orders`;
- Não realizar limpeza ou transformação dos dados;
- Apenas observar, agregar e descrever os dados existentes.

In [4]:
query_overview = """
SELECT
    COUNT(*) AS total_linhas,
    MIN(created_at) AS data_minima,
    MAX(created_at) AS data_maxima,
    MIN(total) AS valor_minimo,
    MAX(total) AS valor_maximo,
    ROUND(AVG(total), 2) AS valor_medio
FROM orders;
"""

overview = conn.execute(query_overview).df()

overview

,total_linhas,data_minima,data_maxima,valor_minimo,valor_maximo,valor_medio
0,48998,2020-01-01 01:19:28,2026-12-31 23:43:09,32.62,127262.02,28704.99


In [5]:
query_nulls = """
SELECT
    SUM(CASE WHEN id IS NULL THEN 1 ELSE 0 END) AS id_nulos,
    SUM(CASE WHEN order_number IS NULL THEN 1 ELSE 0 END) AS order_number_nulos,
    SUM(CASE WHEN channel IS NULL THEN 1 ELSE 0 END) AS channel_nulos,
    SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS customer_id_nulos,
    SUM(CASE WHEN salesperson_id IS NULL THEN 1 ELSE 0 END) AS salesperson_id_nulos,
    SUM(CASE WHEN location_id IS NULL THEN 1 ELSE 0 END) AS location_id_nulos,
    SUM(CASE WHEN status IS NULL THEN 1 ELSE 0 END) AS status_nulos,
    SUM(CASE WHEN subtotal IS NULL THEN 1 ELSE 0 END) AS subtotal_nulos,
    SUM(CASE WHEN discount_amount IS NULL THEN 1 ELSE 0 END) AS discount_amount_nulos,
    SUM(CASE WHEN total IS NULL THEN 1 ELSE 0 END) AS total_nulos,
    SUM(CASE WHEN placed_at IS NULL THEN 1 ELSE 0 END) AS placed_at_nulos,
    SUM(CASE WHEN created_at IS NULL THEN 1 ELSE 0 END) AS created_at_nulos,
    SUM(CASE WHEN updated_at IS NULL THEN 1 ELSE 0 END) AS updated_at_nulos
FROM orders;
"""

null_analysis = conn.execute(query_nulls).df()

null_analysis

,id_nulos,order_number_nulos,channel_nulos,customer_id_nulos,salesperson_id_nulos,location_id_nulos,status_nulos,subtotal_nulos,discount_amount_nulos,total_nulos,placed_at_nulos,created_at_nulos,updated_at_nulos
0,0.0,0.0,0.0,0.0,24131.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [6]:
query_salesperson_nulls = """
SELECT
    channel,
    COUNT(*) AS total_pedidos,
    SUM(CASE WHEN salesperson_id IS NULL THEN 1 ELSE 0 END) AS salesperson_nulo
FROM orders
GROUP BY channel
ORDER BY total_pedidos DESC;
"""

salesperson_nulls = conn.execute(query_salesperson_nulls).df()

salesperson_nulls

,channel,total_pedidos,salesperson_nulo
0,ecommerce,34342,24131.0
1,pos,14656,0.0


In [7]:
query_outliers = """
WITH quartis AS (
    SELECT
        PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY total) AS q1,
        PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY total) AS q3
    FROM orders
),
limites AS (
    SELECT
        q1,
        q3,
        q3 - q1 AS iqr,
        q3 + 1.5 * (q3 - q1) AS limite_superior
    FROM quartis
)
SELECT
    q1,
    q3,
    iqr,
    limite_superior,
    (
        SELECT COUNT(*)
        FROM orders
        WHERE total > limites.limite_superior
    ) AS potenciais_outliers
FROM limites;
"""

outlier_analysis = conn.execute(query_outliers).df()

outlier_analysis

,q1,q3,iqr,limite_superior,potenciais_outliers
0,13171.235,40941.8825,27770.6475,82597.85375,452


# Questão 1.3 — Interpretação

A tabela `orders` apresenta boa consistência inicial para análises exploratórias, mas alguns pontos devem ser validados antes de seu uso em análises mais avançadas. Na coluna `total`, foram identificados **452 registros acima do limite superior de aproximadamente R$ 82.597,85**, calculado pelo método do intervalo interquartil (IQR). Esses valores devem ser considerados **potenciais outliers**, mas não necessariamente erros, já que pedidos de alto valor podem ser compatíveis com o contexto do varejo náutico.

Em relação à completude dos dados, a principal ocorrência de valores ausentes está em `salesperson_id`, com **24.131 registros nulos**. Esses casos aparecem no canal `ecommerce`, enquanto os pedidos realizados via `pos` não apresentam valores nulos nessa coluna, indicando que a ausência pode estar relacionada à própria regra operacional do canal digital. Ainda assim, essa hipótese deve ser validada com a regra de negócio antes de qualquer tratamento.

Dessa forma, considero que a tabela `orders` possui **boa qualidade para análises descritivas em nível de pedido**, desde que os valores extremos e a ausência de vendedor sejam compreendidos e documentados. Para análises mais completas, como rentabilidade por produto, comportamento de clientes, devoluções ou previsão de demanda, a tabela não é suficiente isoladamente e deverá ser relacionada às demais tabelas do modelo, como `order_items`, `customers`, `products` e outras bases pertinentes.
